# Moment 4: distance_gradient_slope

The lightweight one -- a documentation and unit-reconciliation task, not a microdata computation.
No survey in `Data/` reports employment binned by kilometre-distance from a CBD (confirmed during
the literature-verification pass, `paper/notes/literature-verification.md` section 5); the only
usable source is Baez and Kshirsagar (2026), World Bank Policy Research Working Paper 11285,
Table 5b.

## The unit mismatch this moment already survived once

`DECISIONS.md` records this as a locked moment that was originally sourced to the *wrong* table:
Baez and Kshirsagar's Table 1 (population density by kilometre band) was cited in early drafts as
"employment rate by distance band," but their actual employment result -- Table 5b -- is a
continuous coefficient of employment share on a **within-city, population-weighted percentile
rank of network travel time**, not a km-distance gradient. D6 already settled *how* this paper is
used (digitised from the published table, not reproduced from `RR_ZAF_2025_490`, which needs
licensed Stata 18 MP); this notebook settles *which number*, since the source reports three
separate city-specific coefficients, not one.

## Picking a value: mean across cities, not one metro

Table 5b (p.38 printed), column (1) of each city's pair, reports a 10-percentile increase in
travel-time rank corresponding to a *change* in employment share of -4.9 percentage points in
Johannesburg, -3.7 in Cape Town, and -6.5 in eThekwini -- negative, since a higher travel-time-
rank percentile means further from the business district and lower employment share. The model
is a single stylised city, not any one of these three specific metros, so picking one city over
the other two would be an arbitrary choice the model has no basis for -- the mean across all
three, -5.0333 pp per 10 percentile points, is used instead.

**Corrected 2026-08-17.** An earlier version of this notebook recorded the magnitude of each
city's coefficient but dropped the sign, storing a positive mean (+5.0333) as though a higher
travel-time-rank percentile *raised* employment share. Caught by an independent verification
pass, confirmed here by extracting the PDF's own text (`pypdf`, page 39 of 50) rather than
trusting the earlier transcription: the table's row reads "Travel Time -0.49*** -0.46***
-0.37*** -0.21*** -0.65*** -0.65***" under "Table 5b: Spatial Correlates of Share of the Adult
Population Employed" -- every coefficient is negative. See DECISIONS.md, "An independent
verification found distance_gradient_slope's sign and dating both wrong".

## The "standard error" here is cross-city spread, not a sampling SE, and is labelled as such

The source paper reports t-statistics for these coefficients, not standard errors, so the
sampling SE for each city is reconstructed as |coefficient| / |t-statistic| and printed below --
kept separate from `moments.csv`'s `standard_error`, since these three numbers describe
within-city sampling uncertainty, not the between-city heterogeneity a single stylised-city
model needs an uncertainty scale for. `moments.csv`'s `standard_error` is instead the sample
standard deviation across the three cities' point estimates (1.40): real city-to-city
heterogeneity, not a sampling error, and labelled as exactly that in both the notebook and the
`source` column rather than left to be mistaken for one.

## The measurement period is 2011, not the paper's 2026 publication year

The underlying regression links 2011 census sub-place geography (Section 3.1: "the latest
census data available at the sub-place level across South Africa's major cities is from 2011")
to spatial tax employment data. `moments.csv`'s `period` field records when the data was
measured, not when the paper reporting it was published -- corrected from "2026" to "2011"
alongside the sign fix.

In [ ]:
import statistics
from pathlib import Path

import pandas as pd

# Baez and Kshirsagar (2026), WB WP 11285, Table 5b (p.38 printed), column (1) of each city's
# pair (no population-density control, matching the model's own single-channel distance
# measure): percentage-point CHANGE in employment share per 10-percentile increase in
# within-city travel-time rank to the nearest business district. Negative -- a higher travel-
# time-rank percentile (further/slower from the business district) means LOWER employment
# share. Verified directly against the PDF (pypdf text extraction, page 39 of 50): row
# "Travel Time -0.49*** -0.46*** -0.37*** -0.21*** -0.65*** -0.65***" under "Table 5b: Spatial
# Correlates of Share of the Adult Population Employed".
CITY_COEFFICIENTS = {
    "Johannesburg": -4.9,
    "Cape Town": -3.7,
    "eThekwini": -6.5,
}

# The paper reports t-statistics, not standard errors, for these coefficients (same table,
# column 1): Johannesburg -4.44, Cape Town -4.38, eThekwini -9.80. Reconstructed sampling SE
# per city = |coefficient| / |t-statistic| -- printed for the record, kept separate from the
# calibration uncertainty scale below: these three sampling SEs describe within-city
# estimation uncertainty, not the between-city heterogeneity a single stylised-city model
# actually needs a target uncertainty for.
CITY_T_STATISTICS = {
    "Johannesburg": -4.44,
    "Cape Town": -4.38,
    "eThekwini": -9.80,
}
reconstructed_sampling_se = {
    city: abs(CITY_COEFFICIENTS[city]) / abs(CITY_T_STATISTICS[city]) for city in CITY_COEFFICIENTS
}

mean_slope = statistics.mean(CITY_COEFFICIENTS.values())
cross_city_sd = statistics.stdev(CITY_COEFFICIENTS.values())

print("city coefficients (pp per 10 percentile points):", CITY_COEFFICIENTS)
print("reconstructed per-city sampling SE (|coefficient| / |t-statistic|):")
for city, se in reconstructed_sampling_se.items():
    print(f"  {city}: {CITY_COEFFICIENTS[city]} / {CITY_T_STATISTICS[city]} -> SE={se:.4f}")
print(f"mean slope: {mean_slope:.4f}")
print(
    f"cross-city sample SD (calibration uncertainty scale, not a sampling SE): {cross_city_sd:.4f}"
)

In [ ]:
moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "distance_gradient_slope"
moments.loc[row, "value"] = round(mean_slope, 4)
moments.loc[row, "standard_error"] = round(cross_city_sd, 4)
# The underlying regression uses 2011 census sub-place geography (the latest South African
# census available at sub-place level -- Section 3.1 of the paper) linked to spatial tax
# employment data; 2026 is the paper's publication year, not the data's measurement period.
moments.loc[row, "period"] = "2011"
moments.loc[row, "source"] = (
    "Baez, J. and Kshirsagar, V. (2026). South Africa's Fragmented Cities: The Unequal "
    "Burden of Labor Market Frictions. World Bank Policy Research Working Paper 11285. "
    "Table 5b (p.38 printed), column (1) of each city pair -- mean of Johannesburg (-4.9), "
    "Cape Town (-3.7) and eThekwini (-6.5) pp employment-share change per 10-percentile "
    "increase in travel-time rank; negative because higher travel-time rank means lower "
    "employment share. Measurement period is 2011 (sub-place census geography, Section 3.1), "
    "not the paper's 2026 publication year. standard_error is the cross-city sample SD, a "
    "calibration uncertainty scale reflecting city heterogeneity, not a reported sampling SE "
    "-- see notebook 04 for the reconstructed per-city sampling SEs kept separately."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]